# Credit Risk Detection — Full MLOps Pipeline (University GPU Cluster)

**University MLOps Group 10**

This notebook is identical in structure to `colab_full_pipeline.ipynb` but uses a configurable `REPO_ROOT` path instead of hardcoded Colab paths. Run it on any university GPU cluster (SLURM, PBS, or interactive session).

---
## Section 1: Setup

Check the GPU, set paths, install dependencies, and configure API keys.

In [ ]:
# Verify GPU availability
!nvidia-smi

In [ ]:
import os

# ── CONFIGURE THESE TWO LINES ────────────────────────────────────────────────
# Absolute path to the cloned repo on the cluster.
# Examples:
#   REPO_ROOT = os.path.expanduser("~/MLOps-group-10")
#   REPO_ROOT = "/scratch/s1234567/MLOps-group-10"
REPO_ROOT = os.path.expanduser("~/MLOps-group-10")

# API keys — if you already exported them in your shell (e.g. in ~/.bashrc)
# leave these as empty strings and the os.environ.get() fallback below handles it.
HF_TOKEN      = ""   # or: "hf_xxxx"
WANDB_API_KEY = ""   # or: "your_wandb_key"
# ─────────────────────────────────────────────────────────────────────────────

# Use shell exports if the variables above are left blank
HF_TOKEN      = HF_TOKEN      or os.environ.get("HF_TOKEN", "")
WANDB_API_KEY = WANDB_API_KEY or os.environ.get("WANDB_API_KEY", "")

if not HF_TOKEN:
    print("WARNING: HF_TOKEN is not set. Model download from HuggingFace will fail.")
if not WANDB_API_KEY:
    print("WARNING: WANDB_API_KEY is not set. W&B logging will be disabled.")

os.environ["HF_TOKEN"]               = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"]          = WANDB_API_KEY

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"HF_TOKEN  : {'set' if HF_TOKEN else 'NOT SET'}")
print(f"W&B key   : {'set' if WANDB_API_KEY else 'NOT SET'}")

In [ ]:
# Clone or update the repo — edit the URL to your actual GitHub repo.
GITHUB_REPO_URL = "https://github.com/pankti0/MLOps-group-10"

if not os.path.isdir(REPO_ROOT):
    print(f"Cloning {GITHUB_REPO_URL} -> {REPO_ROOT} ...")
    !git clone {GITHUB_REPO_URL} {REPO_ROOT}
    print("Clone complete.")
else:
    print(f"Repo already exists at {REPO_ROOT}, pulling latest...")
    !git -C {REPO_ROOT} pull

In [ ]:
# Install dependencies.
# On most clusters pip installs into your user site-packages (~/.local).
# If the cluster uses modules/conda, activate your environment first.
print("Installing dependencies...")
!pip install -q -r {REPO_ROOT}/requirements.txt
!pip install -q bitsandbytes>=0.41.0   # ensure CUDA-enabled build
print("Done.")

In [ ]:
# Change into the repo root so all relative paths in scripts resolve correctly.
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")
!ls -la

---
## Section 2: Weights & Biases Login

In [ ]:
import wandb

# WANDB_API_KEY is already in the environment — login is non-interactive.
wandb.login()
print(f"W&B version: {wandb.__version__}")

---
## Section 3: Verify Pre-Processed Data

In [ ]:
import os
import pandas as pd

required_paths = {
    "Labels CSV"    : f"{REPO_ROOT}/data/labels/company_labels.csv",
    "FAISS index"   : f"{REPO_ROOT}/data/embeddings/faiss.index",
    "Chunk metadata": f"{REPO_ROOT}/data/embeddings/chunk_metadata.json",
    "Processed dir" : f"{REPO_ROOT}/data/processed",
}

all_ok = True
for name, path in required_paths.items():
    exists = os.path.isfile(path) or os.path.isdir(path)
    status = "OK" if exists else "MISSING"
    if not exists:
        all_ok = False
    print(f"  [{status:7s}] {name}: {path}")

if all_ok:
    print("\nAll required files present.")
else:
    print("\nSome files are missing — check your data pipeline.")

In [ ]:
from IPython.display import display

labels_df = pd.read_csv(f"{REPO_ROOT}/data/labels/company_labels.csv")

def style_risk(val):
    colors = {"low": "background-color:#c6efce;color:#006100",
               "medium": "background-color:#ffeb9c;color:#9c6500",
               "high": "background-color:#ffc7ce;color:#9c0006"}
    return colors.get(str(val).lower(), "")

display(
    labels_df.style
    .applymap(style_risk, subset=["risk_category"])
    .set_caption(f"Company Labels ({len(labels_df)} companies)")
)

In [ ]:
import json
import faiss

index = faiss.read_index(f"{REPO_ROOT}/data/embeddings/faiss.index")
with open(f"{REPO_ROOT}/data/embeddings/chunk_metadata.json") as f:
    meta = json.load(f)

print(f"FAISS index: {index.ntotal} vectors, dimension={index.d}")
print(f"Chunk metadata: {len(meta)} entries")

---
## Section 4: Baseline Inference

Direct Mistral-7B prompting — no retrieval, no fine-tuning.

## ⚡️ Standardized Stratified K-Fold Cross-Validation

All experiments now use **stratified k-fold cross-validation** for robust, fair, and reproducible evaluation.

- Use the `--fold` and `--split` arguments in all inference commands.
- Results are saved as `results/{approach}_fold{N}_{split}.csv`.
- Loop over all folds for full evaluation.

In [ ]:
# Run baseline inference for all folds (1-5) on the test split
for fold in range(1, 6):
    print(f"Running baseline inference for fold {fold} (test split)...")
    !python3 scripts/run_inference_all.py --approach baseline --fold {fold} --split test

In [ ]:
import os
import glob
import pandas as pd
from IPython.display import display

def style_risk_level(val):
    color = {
        "low": "background-color: #c6efce; color: #006100",
        "medium": "background-color: #ffeb9c; color: #9c6500",
        "high": "background-color: #ffc7ce; color: #9c0006",
    }.get(str(val).lower(), "")
    return color

baseline_pattern = f"{REPO_ROOT}/data/results/baseline_fold*_test.csv"
baseline_files = glob.glob(baseline_pattern)

if baseline_files:
    baseline_df = pd.concat([pd.read_csv(f) for f in baseline_files], ignore_index=True)
    display_cols = [col for col in ["ticker", "company_name", "predicted_score", "predicted_label", "risk_level"] if col in baseline_df.columns]

    print(f"Baseline predictions: {len(baseline_df)} rows (cross-validation)")
    display(
        baseline_df[display_cols].style
        .applymap(style_risk_level, subset=["risk_level"] if "risk_level" in baseline_df.columns else [])
        .format({"predicted_score": "{:.1f}"})
        .set_caption("Baseline Inference Results (Cross-Validation)")
    )

    labels_df = pd.read_csv(f"{REPO_ROOT}/data/labels/company_labels.csv")
    merged = baseline_df.merge(labels_df, on="ticker", suffixes=("_pred", "_true"))
    accuracy = (merged["predicted_label"] == merged["label"]).mean()
    print(f"Accuracy: {accuracy:.2%}")
else:
    print(f"No baseline prediction files found matching {baseline_pattern}.")

---
## Section 5: RAG Inference

Mistral-7B with FAISS retrieval — relevant 10-K passages are prepended to the prompt.

- The FAISS index (built from chunked 10-K text) is used to retrieve the most relevant passages for each company.
- Retrieved context is prepended to the prompt before calling Mistral-7B.

In [ ]:
for fold in range(1, 6):
    print(f"Running RAG inference for fold {fold} (test split)...")
    !python3 scripts/run_inference_all.py --approach rag --fold {fold} --split test

In [ ]:
rag_pattern = f"{REPO_ROOT}/data/results/rag_fold*_test.csv"
rag_files = glob.glob(rag_pattern)

if rag_files:
    rag_df = pd.concat([pd.read_csv(f) for f in rag_files], ignore_index=True)
    display_cols = [col for col in ["ticker", "company_name", "predicted_score", "predicted_label", "risk_level"] if col in rag_df.columns]

    print(f"RAG predictions: {len(rag_df)} rows (cross-validation)")
    display(
        rag_df[display_cols].style
        .applymap(style_risk_level, subset=["risk_level"] if "risk_level" in rag_df.columns else [])
        .format({"predicted_score": "{:.1f}"})
        .set_caption("RAG Inference Results (Cross-Validation)")
    )

    labels_df = pd.read_csv(f"{REPO_ROOT}/data/labels/company_labels.csv")
    merged = rag_df.merge(labels_df, on="ticker", suffixes=("_pred", "_true"))
    accuracy = (merged["predicted_label"] == merged["label"]).mean()
    print(f"Accuracy: {accuracy:.2%}")
else:
    print(f"No RAG prediction files found matching {rag_pattern}.")

---
## Section 6: Generate Fine-Tune Data

The fine-tuning data is already committed to the repo and should be present.
This cell re-generates it only if the files are missing or you want a fresh split.
The script creates fold-based instruction-tuning examples from the 10-K sections and labels.

In [ ]:
import os

# Regenerate all folds (run this once after updating your data)
os.system("python3 scripts/generate_finetune_data.py")

In [ ]:
for fold in range(1, 4):  # Use range(1, 6) for 5 folds
    print(f"\nFold {fold}:")
    for split in ["train", "val", "test"]:
        path = f"{REPO_ROOT}/data/finetune/fold_{fold}/{split}.jsonl"
        if os.path.isfile(path):
            with open(path) as f:
                n = sum(1 for line in f if line.strip())
            print(f"  {split:5s}: {n} examples  ({path})")
        else:
            print(f"  {split:5s}: FILE MISSING")

In [ ]:
import json

sample_path = f"{REPO_ROOT}/data/finetune/fold_1/train.jsonl"
if os.path.isfile(sample_path):
    with open(sample_path) as f:
        first_example = json.loads(f.readline())
    print("Sample training example:")
    print(json.dumps(first_example, indent=2)[:1000])
else:
    print(f"File not found: {sample_path}")

---
## Section 7: LoRA Fine-Tuning

Fine-tune Mistral-7B using **LoRA (Low-Rank Adaptation)** with 4-bit quantization via QLoRA.

We found that `lora_r32` performs best so we train it across all 5 folds for the final evaluation.

| Config | Rank `r` | Alpha | Target modules |
|--------|----------|-------|---------------|
| `lora_config_r32.yaml` | 32 | 64 | q, k, v, o, up, down |

In [ ]:
!python3 scripts/train_lora.py --fold 1 --config configs/lora_config_r32.yaml

In [ ]:
!python3 scripts/train_lora.py --fold 2 --config configs/lora_config_r32.yaml

In [ ]:
!python3 scripts/train_lora.py --fold 3 --config configs/lora_config_r32.yaml

In [ ]:
!python3 scripts/train_lora.py --fold 4 --config configs/lora_config_r32.yaml

In [ ]:
!python3 scripts/train_lora.py --fold 5 --config configs/lora_config_r32.yaml

In [ ]:
# Verify adapters were saved for all folds
for fold in range(1, 6):
    adapter_dir = f"{REPO_ROOT}/data/models/lora_adapter_r32/fold_{fold}/lora_config_r32/final_adapter"
    if os.path.isdir(adapter_dir):
        files_list = os.listdir(adapter_dir)
        has_weights = any(f.endswith(".bin") or f.endswith(".safetensors") for f_name in files_list for f in [f_name])
        print(f"  Fold {fold}: OK ({len(files_list)} files)")
    else:
        print(f"  Fold {fold}: MISSING — {adapter_dir}")

In [ ]:
# Pull training loss curves from W&B
try:
    import wandb
    import matplotlib.pyplot as plt

    api = wandb.Api()
    runs = api.runs("credit-risk-detection", filters={"display_name": "lora-r32"})
    runs = sorted(runs, key=lambda r: r.created_at, reverse=True)

    if runs:
        run = runs[0]
        history = run.history(keys=["train/loss", "eval/loss"], pandas=True)

        if not history.empty:
            fig, ax = plt.subplots(figsize=(8, 4))
            if "train/loss" in history.columns:
                ax.plot(history["train/loss"].dropna(), label="Train loss")
            if "eval/loss" in history.columns:
                ax.plot(history["eval/loss"].dropna(), label="Val loss")
            ax.set_xlabel("Step")
            ax.set_ylabel("Loss")
            ax.set_title("LoRA r=32 Fine-Tuning Loss Curve")
            ax.legend()
            plt.tight_layout()
            plt.savefig(f"{REPO_ROOT}/data/results/lora_loss_curve.png", dpi=150)
            plt.show()
            print(f"Loss curve saved. Run: {run.name} | URL: {run.url}")
        else:
            print("No loss history found in W&B run.")
    else:
        print("No W&B runs found. Check your W&B dashboard manually.")
except Exception as e:
    print(f"Could not fetch W&B loss curve: {e}")

---
## Section 8: LoRA Inference

Run inference using the **LoRA fine-tuned adapter** (r=32) across all 5 folds.

In [ ]:
import os
import subprocess

LORA_CONFIGS = {
    "lora_r32": "data/models/lora_adapter_r32",
}

folds = [1, 2, 3, 4, 5]

for approach, base_dir in LORA_CONFIGS.items():
    for fold in folds:
        config_dir = os.path.join(base_dir, f"fold_{fold}", "lora_config_r32")
        final_adapter_dir = os.path.join(config_dir, "final_adapter")

        if os.path.isfile(os.path.join(final_adapter_dir, "adapter_config.json")):
            adapter_path = final_adapter_dir
        elif os.path.isfile(os.path.join(config_dir, "adapter_config.json")):
            adapter_path = config_dir
        else:
            print(f"[SKIP] {approach} fold {fold}: adapter_config.json not found")
            continue

        print(f"\nRunning inference for {approach} — fold {fold}")
        result = subprocess.run([
            "python3", "scripts/run_inference_all.py",
            "--approach", approach,
            "--adapter-path", adapter_path,
            "--fold", str(fold),
            "--split", "test"
        ], capture_output=True, text=True)
        print("STDOUT:\n", result.stdout)
        if result.stderr:
            print("STDERR:\n", result.stderr)
        if result.returncode != 0:
            print(f"[ERROR] Exit code {result.returncode}")

print("LoRA inference complete.")

In [ ]:
import os
import subprocess

approach = "lora_r32_rag"
base_dir = "data/models/lora_adapter_r32"
folds = [1, 2, 3, 4, 5]

for fold in folds:
    config_dir = os.path.join(base_dir, f"fold_{fold}", "lora_config_r32")
    final_adapter_dir = os.path.join(config_dir, "final_adapter")

    if os.path.isfile(os.path.join(final_adapter_dir, "adapter_config.json")):
        adapter_path = final_adapter_dir
    elif os.path.isfile(os.path.join(config_dir, "adapter_config.json")):
        adapter_path = config_dir
    else:
        print(f"[SKIP] {approach} fold {fold}: adapter_config.json not found")
        continue

    print(f"\nRunning inference for {approach} — fold {fold}")
    result = subprocess.run([
        "python3", "scripts/run_inference_all.py",
        "--approach", approach,
        "--adapter-path", adapter_path,
        "--fold", str(fold),
        "--split", "test"
    ], capture_output=True, text=True)
    print("STDOUT:\n", result.stdout)
    if result.stderr:
        print("STDERR:\n", result.stderr)
    if result.returncode != 0:
        print(f"[ERROR] Exit code {result.returncode}")

print("RAG+LoRA inference complete.")

---
## Section 9: Full Evaluation

Run the complete evaluation pipeline comparing all approaches:

| Approach | Description |
|----------|-------------|
| Baseline | Direct Mistral-7B prompting |
| RAG | Mistral-7B + FAISS retrieval |
| LoRA r=32 | Fine-tuned Mistral-7B (k-fold) |
| Altman Z-Score | Classical financial model (benchmark) |

Outputs:
- `data/results/metrics_summary.csv`
- `data/results/roc_curves.png`

In [ ]:
# Run full evaluation pipeline across all folds
!python3 scripts/evaluate_folds.py

In [ ]:
import pandas as pd
from IPython.display import display
import os

metrics_path = f"{REPO_ROOT}/data/results/metrics_summary.csv"

if os.path.isfile(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    numeric_cols = metrics_df.select_dtypes(include="number").columns.tolist()

    def highlight_best(s):
        if s.name not in numeric_cols:
            return [""] * len(s)
        is_best = s == s.max()
        return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_best]

    display(
        metrics_df.style
        .apply(highlight_best)
        .format({col: "{:.4f}" for col in numeric_cols})
        .set_caption("Comparison of Credit Risk Detection Approaches")
        .set_table_styles([{
            "selector": "th",
            "props": [("background-color", "#343a40"), ("color", "white"), ("font-weight", "bold")]
        }])
    )
else:
    print(f"Metrics summary not found at {metrics_path}. Check evaluate_folds.py output above.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

roc_path = f"{REPO_ROOT}/data/results/roc_curves.png"

if os.path.isfile(roc_path):
    img = mpimg.imread(roc_path)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("ROC Curves — All Approaches", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print(f"ROC curve image not found at {roc_path}.")

In [ ]:
# Show a per-company prediction comparison across all approaches
import pandas as pd
from IPython.display import display
import os

RESULTS_DIR = f"{REPO_ROOT}/data/results"
LABELS_PATH = f"{REPO_ROOT}/data/labels/company_labels.csv"

labels = pd.read_csv(LABELS_PATH)[["ticker", "label", "risk_category"]].rename(
    columns={"label": "true_label", "risk_category": "true_risk"}
)

comparison = labels.copy()

for approach in ["baseline", "rag", "lora_r32", "altman_zscore"]:
    pred_path = os.path.join(RESULTS_DIR, f"{approach}_predictions.csv")
    if os.path.isfile(pred_path):
        df = pd.read_csv(pred_path)[["ticker", "predicted_label", "predicted_score"]]
        df = df.rename(columns={
            "predicted_label": f"{approach}_pred",
            "predicted_score" : f"{approach}_score",
        })
        comparison = comparison.merge(df, on="ticker", how="left")

def highlight_correct(val):
    if val is True:
        return "background-color: #d4edda"
    elif val is False:
        return "background-color: #f8d7da"
    return ""

for approach in ["baseline", "rag", "lora_r32", "altman_zscore"]:
    col = f"{approach}_pred"
    if col in comparison.columns:
        comparison[f"{approach}_correct"] = (comparison[col] == comparison["true_label"])

correct_cols = [c for c in comparison.columns if c.endswith("_correct")]
display(
    comparison.style
    .map(highlight_correct, subset=correct_cols)
    .set_caption("Prediction Comparison (green = correct, red = wrong)")
)

---
## Section 10: Results Location

On the cluster, results are saved to disk — use `scp` or the cluster's file transfer portal to retrieve them.

In [ ]:
import os

print("=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)

output_files = {
    "Metrics summary"      : f"{REPO_ROOT}/data/results/metrics_summary.csv",
    "ROC curves"           : f"{REPO_ROOT}/data/results/roc_curves.png",
    "LoRA loss curve"      : f"{REPO_ROOT}/data/results/lora_loss_curve.png",
    "Altman Z-Score"       : f"{REPO_ROOT}/data/results/altman_zscore_predictions.csv",
    "LoRA adapter r=32"    : f"{REPO_ROOT}/data/models/lora_adapter_r32",
}

# Also check fold result files
for approach in ["baseline", "rag", "lora_r32"]:
    for fold in range(1, 6):
        key = f"{approach} fold {fold}"
        path = f"{REPO_ROOT}/data/results/{approach}_fold{fold}_test.csv"
        output_files[key] = path

print("\nOutput files:")
for name, path in output_files.items():
    exists = os.path.isfile(path) or os.path.isdir(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status:7s}] {name}")

print(f"\nTo copy results to your local machine, run from your laptop:")
print(f"  scp -r <username>@<cluster-hostname>:{REPO_ROOT}/data/results/ ./results/")
print(f"  scp -r <username>@<cluster-hostname>:{REPO_ROOT}/data/models/ ./models/")
print(f"\nW&B Dashboard: https://wandb.ai -> project: credit-risk-detection")
print("=" * 60)